### Setting up

In [ ]:
# Add SSAS dlls to path
from sys import path
path.append('./Microsoft.AnalysisServices.Tabular')
path.append('./Microsoft.AnalysisServices.AdomdClient')

# pip install Office365-REST-Python-Client
from office365.sharepoint.client_context import ClientContext
from office365.runtime.auth.client_credential import ClientCredential
from office365.sharepoint.folders.folder import Folder

# Import packages
import os
import csv
import shutil
import nest_asyncio
import pandas as pd
from time import time
from typing import Dict, List
from auth import Auth
from workspace import Workspace
from dataflow import Dataflow
from dataset import Dataset
from pyadomd import Pyadomd

import asyncio
nest_asyncio.apply()

# Paths
ROOT_PATH = './data/data_hub'
SHAREPOINT_ROOT_PATH = 'Shared Documents/General/2. Arquivos/8. Data Hub'

# Tenant/app settings
TENANT_ID = os.environ.get('TENANT_ID', '')
CLIENT_ID = os.environ.get('CLIENT_ID', '')
CLIENT_SECRET = os.environ.get('CLIENT_SECRET', '')

# Site settings
CLIENT_ID_SHAREPOINT = os.environ.get('CLIENT_ID_SHAREPOINT', '')
CLIENT_SECRET_SHAREPOINT = os.environ.get('CLIENT_SECRET_SHAREPOINT', '')
HOST = 'anheuserbuschinbev.sharepoint.com'
SITE = 'PeopleTech-DataAnalyticsTeam'
SITE_URL = f'https://{HOST}/sites/{SITE}'
target_folder_url = f'Shared Documents/General/2. Arquivos/8. Data Hub'

In [ ]:
# If ROOT_PATH exists, removes it and create a fresh one.
if os.path.isdir(ROOT_PATH):
    shutil.rmtree(ROOT_PATH)

os.makedirs(ROOT_PATH)

In [ ]:
# PBI API Authentication (get bearer token)
auth = Auth(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
token = auth.get_token()

# Sharepoint (sign with service principal)
client_credentials = ClientCredential(CLIENT_ID_SHAREPOINT, CLIENT_SECRET_SHAREPOINT)
ctx = ClientContext(SITE_URL).with_credentials(client_credentials)
ctx.web.get_folder_by_server_relative_path(SHAREPOINT_ROOT_PATH).delete_object().execute_query()
target_folder = ctx.web.ensure_folder_path(SHAREPOINT_ROOT_PATH).execute_query()

# Initializing objects
workspace = Workspace(token)
dataset = Dataset(token)
dataflow = Dataflow(token)

### Getting workspaces data

In [ ]:
# Ignore workspaces list
ignore_workspaces = [
    'Brewdat Dev - GHQ - People - PSI'
]

# Get list of workspaces
workspaces_list = workspace.list_workspaces()
workspaces_data= workspaces_list.get('content', [])
df_workspaces = pd.DataFrame(workspaces_data)

# Create columns
df_workspaces['premium_link'] = 'powerbi://api.powerbi.com/v1.0/myorg/' + df_workspaces['name']
df_workspaces['webUrl'] = 'https://app.powerbi.com/groups/' + df_workspaces['id']

# Remove without capacityId and from ignore list
df_workspaces = df_workspaces[(~df_workspaces['capacityId'].isna())&(~df_workspaces['name'].isin(ignore_workspaces))]

# Sort and save
df_workspaces.sort_values(by=['name'], inplace=True)
df_workspaces.reset_index(inplace=True, drop=True)
file_name = 'workspaces.csv'
file_path = f'{ROOT_PATH}/{file_name}'
df_workspaces.to_csv(file_path, sep='@', index=False, encoding='utf-8-sig')

with open(file_path, 'rb') as file:
    file_content = file.read()
    
target_file = target_folder.upload_file(file_name, file_content).execute_query()

print('Total columns:', len(df_workspaces.columns))
df_workspaces.describe()

### Getting datasets data

In [ ]:
# Ignore datasets and reports that are not used for our Data Hub
ignore_list = [
    'Report Usage Metrics Model',
    'Report Usage Metrics Report',
    'Usage Metrics Report',
    'Dashboard Usage Metrics Model',
    'Access Control Report',
    'Engagement+ -TEST'
]

# Get datasets data
df_datasets_list = []
for workspace_to_search in df_workspaces.to_dict('records'):

    datasets_list = dataset.list_datasets(workspace_id=workspace_to_search['id'])
    datasets_data = datasets_list.get('content', [])
    df = pd.DataFrame(datasets_data)
    df['workspace_id'] = workspace_to_search['id']
    df['workspace_name'] = workspace_to_search['name']
    df_datasets_list.append(df)

    df = None
    del df

df_datasets = pd.concat(df_datasets_list)

# Create columns
df_datasets['premium_link'] = 'powerbi://api.powerbi.com/v1.0/myorg/' + df_datasets['workspace_name']

# Remove untitled and from ignore list
df_datasets = df_datasets[(~df_datasets['name'].isin(ignore_list))&(~df_datasets['name'].str.contains('Untitled'))]

# Apply filters
df_datasets = df_datasets[df_datasets['addRowsAPIEnabled']==False]

# Sort and save
df_datasets.sort_values(by=['workspace_name', 'name'])
df_datasets.reset_index(inplace=True, drop=True)
file_name = 'datasets.csv'
file_path = f'{ROOT_PATH}/{file_name}'
df_datasets.to_csv(file_path, sep='@', index=False, encoding='utf-8-sig')

with open(file_path, 'rb') as file:
    file_content = file.read()
    
target_file = target_folder.upload_file(file_name, file_content).execute_query()

print('Total columns:', len(df_datasets.columns))
df_datasets.describe()

### Getting reports data

In [ ]:
# Get reports data
df_reports_list = []
for workspace_to_search in df_workspaces.to_dict('records'):

    reports_list = workspace.list_reports(workspace_id=workspace_to_search['id'])
    reports_data = reports_list.get('content', [])
    df = pd.DataFrame(reports_data)
    df['workspace_id'] = workspace_to_search['id']
    df['workspace_name'] = workspace_to_search['name']
    df_reports_list.append(df)

    df = None
    del df

df_reports = pd.concat(df_reports_list)

# Apply filters
df_reports = df_reports[(~df_reports['name'].isin(ignore_list))&(~df_reports['name'].str.contains('Untitled'))]

# Sort and save
df_reports.sort_values(by=['workspace_name', 'name'])
file_name = 'reports.csv'
file_path = f'{ROOT_PATH}/{file_name}'
df_reports.to_csv(file_path, sep='@', index=False, encoding='utf-8-sig')

with open(file_path, 'rb') as file:
    file_content = file.read()
    
target_file = target_folder.upload_file(file_name, file_content).execute_query()

print('Total columns:', len(df_reports.columns))
df_reports.describe()

### Getting tables, columns and measures data
##### We get this connecting to the SSAS model for each dataset using XMLA endpoints.

In [ ]:
def get_token():
    
    global expires_in
    global token_obtained_on
    global expires_on
    
    expires_in = 3000
    token_obtained_on = time()
    expires_on = token_obtained_on + expires_in
        
    # PBI API Authentication (get bearer token)
    auth = Auth(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
    token = auth.get_token()

    return token


def save_and_upload_file(df, file_name: str, file_path: str, target_folder: Folder):
    
    # Save to CSV
    df.to_csv(f'{file_path}', sep='@', index=False, encoding='utf-8-sig')
    
    # Upload the CSV to SharePoint
    with open(file_path, 'rb') as file:
        file_content = file.read()

    _ = target_folder.upload_file(file_name, file_content).execute_query()


def get_and_process_data(conn, parameter, query, headers, data, target_folder):
    
    dataset_path = data['dataset_path']
      
    with conn.cursor().execute(query) as query_results:

        result = query_results.fetchall()

    data[f'{parameter}_header'] = headers
    data[f'{parameter}_data'] = result
    df = pd.DataFrame(data[f'{parameter}_data'], columns=data[f'{parameter}_header'])
    df['workspace_name'] = data['workspace_name']
    df['dataset_name'] = data['dataset_name']
    df['dataset_id'] = data['dataset_id']
    file_name = f'{parameter}.csv'
    file_path = f'{dataset_path}/{file_name}'
    
    save_and_upload_file(df, file_name, file_path, target_folder)
                        
                        
async def fetch_data(args: List[str]):
    """
    Get data from a PBI Dataset:
        - Tables;
        - Columns;
        - Measures;
        - Model;
    """
    global counter
    global totals
    global token
    
    if (token == '') | (time() > expires_on):
        print(f'Getting new token on iteration {counter+1}...')

        token = get_token()
        
    workspace_name, dataset_id, dataset_name = args[0], args[1], args[2]
    print(counter, totals, dataset_name)
    
    data = {}
                
    workspace_path = f'{ROOT_PATH}/{workspace_name}'
    dataset_path = f'{workspace_path}/{dataset_name}'

    data['workspace_name'] = workspace_name
    data['dataset_id'] = dataset_id
    data['dataset_name'] = dataset_name
    data['dataset_path'] = dataset_path
    
    # Create folders if they do not exist
    for p in [workspace_path, dataset_path]:
        if not os.path.exists(p):
            os.mkdir(p)      
    
    # Sharepoint target folder (create it if do not exists)
    target_folder_path = f'{SHAREPOINT_ROOT_PATH}/{workspace_name}/{dataset_name}'
    target_folder = ctx.web.ensure_folder_path(target_folder_path).execute_query()     

    # Build the connection string
    connection_string_part1 = f'Provider=MSOLAP.8;Data Source=powerbi://api.powerbi.com/v1.0/myorg/{workspace_name};Initial Catalog={dataset_name};'
    connection_string_part2 = f'User ID=;Password={token};'
    connection_string = connection_string_part1 + connection_string_part2
    
    # If we don't have the files already.
    # This is just for testing, in production this should be removed.
    if  (
              (not os.path.isfile(f'{dataset_path}/tables.csv'))
            | (not os.path.isfile(f'{dataset_path}/columns.csv'))
            | (not os.path.isfile(f'{dataset_path}/measures.csv'))
            | (not os.path.isfile(f'{dataset_path}/models.csv'))
        ):
            
            try:

                with Pyadomd(connection_string) as conn:

                    # Get tables data
                    tables_query = ''' 
                        SELECT [ID] AS [Table ID] ,
                            [Name] AS [Table] ,
                            [IsHidden] ,
                            [IsPrivate] ,
                            [ModifiedTime] AS [Modified Datetime] ,
                            [StructureModifiedTime] AS [Structure Modified Datetime]
                        FROM $SYSTEM.TMSCHEMA_TABLES
                    '''
                    headers = ['Table ID', 'Table', 'IsHidden', 'IsPrivate', 'Modified Datetime', 'Structure Modified Datetime']
                    get_and_process_data(conn, 'tables', tables_query, headers, data, target_folder)


                    # Get columns data
                    columns_query = '''
                        SELECT [ID] AS [Column ID],
                            [TableID] AS [Table ID],
                            [ExplicitName] AS [Column],
                            [InferredName] AS [Column Inferred Name],
                            [ExplicitDataType] AS [Data Type ID],
                            [InferredDataType] AS [Inferred Data Type ID],
                            [DataCategory] AS [Data Category],
                            [SourceColumn] AS [Source Column]
                        FROM $SYSTEM.TMSCHEMA_COLUMNS
                    '''
                    headers = ['Column ID', 'Table ID', 'Column', 'Column Inferred Name', 'Data Type ID', 'Inferred Data Type ID', 'Data Category', 'Source Column']
                    get_and_process_data(conn, 'columns', columns_query, headers, data, target_folder)

                    # Get measures data
                    measures_query = '''
                        SELECT [ID] AS [Measure ID],
                        [TableID] AS [Table ID],
                        [Name],
                        [DataType] AS [Data Type ID],
                        [DisplayFolder] AS [Display Folder],
                        [Description],
                        [Expression]
                        FROM $SYSTEM.TMSCHEMA_MEASURES
                    '''
                    headers = ['Measure ID', 'Table ID', 'Name', 'Data Type ID', 'Display Folder', 'Description', 'Expression']
                    get_and_process_data(conn, 'measures', measures_query, headers, data, target_folder)

                    # # Get models data
                    # models_query = '''
                    #     SELECT *
                    #     FROM $SYSTEM.TMSCHEMA_MODEL
                    # '''
                    # headers = [
                    #     'ID', 'Name', 'Description', 'StorageLocation', 'DefaultMode', 'DefaultDataView', 
                    #     'Culture', 'Collation', 'ModifiedTime', 'StructureModifiedTime', 'Version', 'DataAccessOptions', 
                    #     'DefaultMeasureID', 'DefaultPowerBIDataSourceVersion', 'ForceUniqueNames', 'DiscourageImplicitMeasures', 
                    #     'DataSourceVariablesOverrideBehavior', 'DataSourceDefaultMaxConnections', 'SourceQueryCulture', 'MAttributes', 
                    #     'DiscourageCompositeModels', 'AutomaticAggregationOptions', 'DisableAutoExists', 'workspace_name', 'dataset_name', 'dataset_id']
                    # get_and_process_data(conn, 'models', models_query, headers, data, target_folder)
                        
                fields=[counter, workspace_name, dataset_id, dataset_name, 'Success']
                with open(r'logs.csv', 'a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow(fields)
                    
                counter += 1
            except Exception as e:
                
                fields=[counter, workspace_name, dataset_id, dataset_name, str(e)]
                with open(r'logs.csv', 'a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow(fields)
                
                counter += 1
                # print(f'\rTotal calls (error at): {counter} / {totals}\nError:\n{e}', end=' ')
                # print(f'Error with {workspace_name} - {dataset_id} - {dataset_name}: {str(e)}')


In [ ]:
async def main() -> Dict:
    """
    This is the main function and returns all the data from all datasets:
        - Dataset name and id;
        - Tables;
        - Columns;
        - Measures;
        - Model;

    Returns:
        results (dict): dictionary with data from all datasets.
    """
    

    datasets_id = df_datasets['id'].values.tolist()
    datasets_name = df_datasets['name'].values.tolist()
    workspaces_names = df_datasets['workspace_name'].tolist()
    
    global totals
    totals = len(list(zip(workspaces_names, datasets_id, datasets_name)))

    tasks = []
    for workspace_name, dataset_id, dataset_name in zip(workspaces_names, datasets_id, datasets_name):

        tasks.append(asyncio.ensure_future(fetch_data([workspace_name, dataset_id, dataset_name])))

    results = await asyncio.gather(*tasks)
    
    return results
    
    
if __name__ == '__main__':
    
    counter = 0
    totals = 0
    
    expires_in = 3000
    token_obtained_on = 0
    expires_on = 0
    
    token = ''
    
    results = await main()